# Sistema de Recomendación Multimodal vs Baseline
**MovieLens 1M + TMDB (posters + sinopsis)**

Comparamos:
- **Modelo A (Baseline)**: LightFM solo con ratings
- **Modelo B (Multimodal)**: LightFM con item features = embeddings de texto (sinopsis) + imagen (poster) + géneros

Métrica principal: **NDCG@10** y **Precision@10** sobre el Top-10 recomendado por usuario

## 0. Instalación

In [1]:
!pip install scikit-learn scipy pandas numpy matplotlib seaborn tqdm
!pip install git+https://github.com/daviddavo/lightfm
!pip install recommenders

  Cloning https://github.com/daviddavo/lightfm to /tmp/pip-req-build-739413nr
  Running command git clone --filter=blob:none --quiet https://github.com/daviddavo/lightfm /tmp/pip-req-build-739413nr
  Resolved https://github.com/daviddavo/lightfm to commit f0eb500ead54ab65eb8e1b3890337a7223a35114
  Preparing metadata (setup.py) ... done
  Created wheel for lightfm: filename=lightfm-1.17-cp312-cp312-linux_x86_64.whl size=1099143 sha256=217e9205738814d376f455e9a3e54a82fe2e070f333d1a19a49583b572aeac98
  Stored in directory: /tmp/pip-ephem-wheel-cache-l9shger7/wheels/fd/89/93/70c1e5f378ee5043de89387ee3ef6852ff39e3b9eb44ecc1a3
Successfully built lightfm
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.6/51.6 kB 889.4 kB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.2/47.2 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 355.3/355.3 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━

## 1. Carga de datos

In [ ]:
import numpy as np
import pandas as pd
import scipy.sparse as sp
from sklearn.preprocessing import normalize, MultiLabelBinarizer
from sklearn.decomposition import PCA
from lightfm import LightFM
from lightfm.data import Dataset
from lightfm.evaluation import precision_at_k
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

import kagglehub
kaggle_path = kagglehub.dataset_download("odedgolden/movielens-1m-dataset")

RATINGS_PATH    = f"{kaggle_path}/ratings.dat"

ITEM_TABLE_PATH = "item_table.csv"
TEXT_EMB_PATH   = "embeddings/text_embeddings.npy"
IMG_EMB_PATH    = "embeddings/image_embeddings.npy"

# Ratings
ratings = pd.read_csv(RATINGS_PATH, sep="::", engine="python",
                      names=["user_id", "movie_id", "rating", "timestamp"])

item_table = pd.read_csv(ITEM_TABLE_PATH)

# Embeddings precomputados
text_emb = np.load(TEXT_EMB_PATH)   
img_emb  = np.load(IMG_EMB_PATH)    

print(f"Ratings:        {ratings.shape}")
print(f"Items:          {item_table.shape}")
print(f"Text embeddings:{text_emb.shape}")
print(f"Image embeddings:{img_emb.shape}")

Using Colab cache for faster access to the 'movielens-1m-dataset' dataset.
Ratings:        (1000209, 4)
Items:          (3883, 9)
Text embeddings:(3883, 384)
Image embeddings:(3883, 768)


In [ ]:
def ndcg_at_k(model, interactions, k=10, item_features=None, num_threads=1):
    """NDCG@K manual compatible con cualquier versión de LightFM."""
    interactions_csr = interactions.tocsr()
    n_users = interactions.shape[0]
    n_items = interactions.shape[1]
    ndcgs = []

    for user_id in range(n_users):
        # Ítems relevantes 
        relevant = set(interactions_csr[user_id].indices)
        if len(relevant) == 0:
            continue

        scores = model.predict(user_id, np.arange(n_items), item_features=item_features)
        top_k  = np.argsort(-scores)[:k]

        # DCG
        dcg = sum(
            1.0 / np.log2(rank + 2)
            for rank, item in enumerate(top_k)
            if item in relevant
        )

        # IDCG
        ideal_hits = min(len(relevant), k)
        idcg = sum(1.0 / np.log2(rank + 2) for rank in range(ideal_hits))

        if idcg > 0:
            ndcgs.append(dcg / idcg)

    return np.array(ndcgs)

## 2. Train / Test split por usuario 

Para cada usuario, el 20% de sus ratings más recientes va al test set.  


In [7]:
ratings = ratings.sort_values(["user_id", "timestamp"])

def temporal_split(df, test_frac=0.2):
    train_idx, test_idx = [], []
    for _, grp in df.groupby("user_id"):
        n = len(grp)
        cut = max(1, int(n * test_frac))
        train_idx.extend(grp.index[:-cut])
        test_idx.extend(grp.index[-cut:])
    return df.loc[train_idx].reset_index(drop=True), df.loc[test_idx].reset_index(drop=True)

train_df, test_df = temporal_split(ratings, test_frac=0.2)
print(f"Train: {len(train_df):,} | Test: {len(test_df):,}")


Train: 802,553 | Test: 197,656


## 3. Construcción de matrices de interacción para LightFM

In [8]:
all_users = sorted(ratings["user_id"].unique())
all_items = sorted(item_table["movie_id"].unique())

dataset = Dataset()
dataset.fit(users=all_users, items=all_items)

n_users, n_items = dataset.interactions_shape()
print(f"Usuarios: {n_users} | Ítems: {n_items}")

def build_interactions(df, dataset):
    interactions, weights = dataset.build_interactions(
        [(row.user_id, row.movie_id, row.rating) for row in df.itertuples()]
    )
    return interactions, weights

train_interactions, train_weights = build_interactions(train_df, dataset)
test_interactions,  test_weights  = build_interactions(test_df,  dataset)

print(f"Train matrix: {train_interactions.shape}, nnz={train_interactions.nnz:,}")
print(f"Test  matrix: {test_interactions.shape},  nnz={test_interactions.nnz:,}")

Usuarios: 6040 | Ítems: 3883
Train matrix: (6040, 3883), nnz=802,553
Test  matrix: (6040, 3883),  nnz=197,656


## 4. Item features para el modelo multimodal

Combinamos:
1. **Géneros** (one-hot, 18 dimensiones)
2. **Texto** (sinopsis vía sentence-transformers, 384d → PCA 128d)
3. **Imagen** (poster vía CLIP, 768d → PCA 128d)

Concatenamos y normalizamos → feature matrix de shape (N_items, 18+128+128)

In [ ]:
PCA_DIM = 128

# 4.1 Géneros one-hot
item_table["genres_list"] = item_table["genres"].fillna("").apply(lambda g: g.split("|"))
mlb = MultiLabelBinarizer()
genre_matrix = mlb.fit_transform(item_table["genres_list"]).astype(np.float32)
print(f"Géneros one-hot: {genre_matrix.shape}  |  clases: {mlb.classes_}")

# 4.2 PCA sobre embeddings de texto
text_pca = PCA(n_components=PCA_DIM, random_state=42)
text_reduced = text_pca.fit_transform(text_emb[item_table["emb_idx"].values]).astype(np.float32)
print(f"Text PCA varianza explicada: {text_pca.explained_variance_ratio_.sum():.1%}")

# 4.3 PCA sobre embeddings de imagen 
img_raw = img_emb[item_table["emb_idx"].values]
zero_mask = (img_raw == 0).all(axis=1)
print(f"Películas sin poster (vector cero): {zero_mask.sum()} ({zero_mask.mean():.1%})")

img_pca = PCA(n_components=PCA_DIM, random_state=42)
img_reduced = np.zeros((len(img_raw), PCA_DIM), dtype=np.float32)
img_reduced[~zero_mask] = img_pca.fit_transform(img_raw[~zero_mask])  # solo filas no vacías
print(f"Image PCA varianza explicada: {img_pca.explained_variance_ratio_.sum():.1%}")

# 4.4 Normalizar cada bloque por separado y luego concatenar
genre_norm = normalize(genre_matrix, norm="l2")
text_norm  = normalize(text_reduced, norm="l2")
img_norm   = normalize(img_reduced,  norm="l2")

combined = np.hstack([genre_norm, text_norm, img_norm])
print(f"Feature matrix final: {combined.shape}")  # (3883, 18+128+128)

# 4.5 Construir item_features para LightFM
feature_names = (
    [f"genre_{g}" for g in mlb.classes_] +
    [f"text_{i}"  for i in range(PCA_DIM)] +
    [f"img_{i}"   for i in range(PCA_DIM)]
)

dataset_mm = Dataset()
dataset_mm.fit(users=all_users, items=all_items, item_features=feature_names)

item_features_list = []
for row_idx, item_row in item_table.iterrows():
    feat_dict = {fname: float(combined[row_idx, fi]) for fi, fname in enumerate(feature_names)}
    item_features_list.append((item_row["movie_id"], feat_dict))

item_features_mm = dataset_mm.build_item_features(item_features_list, normalize=False)
print(f"Item features sparse matrix: {item_features_mm.shape}")

train_int_mm, train_w_mm = build_interactions(train_df, dataset_mm)
test_int_mm,  test_w_mm  = build_interactions(test_df,  dataset_mm)

Géneros one-hot: (3883, 18)  |  clases: ['Action' 'Adventure' 'Animation' "Children's" 'Comedy' 'Crime'
 'Documentary' 'Drama' 'Fantasy' 'Film-Noir' 'Horror' 'Musical' 'Mystery'
 'Romance' 'Sci-Fi' 'Thriller' 'War' 'Western']
Text PCA varianza explicada: 82.9%
Películas sin poster (vector cero): 239 (6.2%)
Image PCA varianza explicada: 68.2%
Feature matrix final: (3883, 274)
Item features sparse matrix: (3883, 4157)


## 5. Entrenamiento



In [ ]:
EPOCHS      = 50
NO_COMPONENTS = 128 

# Modelo A: Baseline (solo ratings, sin item features) 
model_baseline = LightFM(
    no_components=NO_COMPONENTS,
    loss="bpr",
    learning_rate=0.03,   
    item_alpha=1e-8,
    user_alpha=1e-6,
    max_sampled=10,
    random_state=42
)
for epoch in tqdm(range(EPOCHS), desc="Baseline"):
    model_baseline.fit_partial(
        train_interactions,
        sample_weight=train_weights,
        num_threads=4,
        epochs=1
    )


Baseline: 100%|██████████| 50/50 [08:06<00:00,  9.74s/it]


In [ ]:
# Modelo A: Multimodal (ratings + item features)

model_multimodal = LightFM(
    no_components=NO_COMPONENTS,
    loss="bpr",
    learning_rate=0.03,   
    item_alpha=1e-8,      
    user_alpha=1e-6,
    max_sampled=10,
    random_state=42
)
for epoch in tqdm(range(EPOCHS), desc="Multimodal"):
    model_multimodal.fit_partial(
        train_int_mm,
        item_features=item_features_mm,
        sample_weight=train_w_mm,
        num_threads=4,
        epochs=1
    )

print("\n✓ Entrenamiento completado")

Multimodal: 100%|██████████| 50/50 [6:19:31<00:00, 455.43s/it]


✓ Entrenamiento completado


## 6. Evaluación — Precision@10 y NDCG@10

In [18]:
K = 10
NUM_THREADS=4

# Modelo A: Baseline
prec_base = precision_at_k(model_baseline, test_interactions, k=K).mean()
ndcg_base = ndcg_at_k(model_baseline, test_interactions, k=K).mean()

# Modelo B: Multimodal
prec_mm = precision_at_k(model_multimodal, test_int_mm, item_features=item_features_mm, k=K, num_threads=NUM_THREADS).mean()
ndcg_mm = ndcg_at_k(model_multimodal, test_int_mm, item_features=item_features_mm, k=K, num_threads=NUM_THREADS).mean()

results = pd.DataFrame({
    "Modelo":        ["Baseline (solo ratings)", "Multimodal (ratings + texto + imagen)"],
    "Precision@10":  [prec_base, prec_mm],
    "NDCG@10":       [ndcg_base, ndcg_mm],
})

print(results.to_string(index=False, float_format="{:.4f}".format))

delta_prec = (prec_mm - prec_base) / prec_base * 100
delta_ndcg = (ndcg_mm - ndcg_base) / ndcg_base * 100
print(f"\nMejora relativa Multimodal vs Baseline:")
print(f"  Precision@10: {delta_prec:+.1f}%")
print(f"  NDCG@10:      {delta_ndcg:+.1f}%")

                               Modelo  Precision@10  NDCG@10
              Baseline (solo ratings)        0.0193   0.0207
Multimodal (ratings + texto + imagen)        0.0167   0.0180

Mejora relativa Multimodal vs Baseline:
  Precision@10: -13.5%
  NDCG@10:      -13.3%
